#### Importing Libraries

In [1]:
# Importing libraries for text feature engineering
import pandas as pd
import numpy as np
import joblib
from pathlib import Path
from scipy.sparse import hstack, save_npz, load_npz
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder

#### Loading the Prepared Datasets

In [2]:
# Defining paths for prepared datasets
processed_dir = Path("../data/processed")
models_dir = Path("../models")

# Loading the validated training and testing datasets
train_df = pd.read_csv(processed_dir / "train.csv")
test_df = pd.read_csv(processed_dir / "test.csv")

# Displaying dataset dimensions
print(f"Training dataset: {train_df.shape}")
print(f"Testing dataset: {test_df.shape}")

# Verifying required columns
required_columns = ["Product_Title", "Category_Label", "Split_Group"]

assert all(column in train_df.columns for column in required_columns)
assert all(column in test_df.columns for column in required_columns)

print("Required columns verified successfully.")

Training dataset: (28919, 8)
Testing dataset: (6391, 8)
Required columns verified successfully.


#### Separating Text Features and Target Labels

In [3]:
# Extracting product titles as text features
X_train_text = train_df["Product_Title"].copy()
X_test_text = test_df["Product_Title"].copy()

# Extracting target labels
y_train = train_df["Category_Label"].copy()
y_test = test_df["Category_Label"].copy()

# Displaying feature and target dimensions
print(f"Training text features: {X_train_text.shape}")
print(f"Testing text features: {X_test_text.shape}")
print(f"Training target labels: {y_train.shape}")
print(f"Testing target labels: {y_test.shape}")

# Verifying that text features contain no missing values
assert X_train_text.notna().all()
assert X_test_text.notna().all()

# Verifying that all test categories exist in training data
assert set(y_test.unique()).issubset(set(y_train.unique()))

print("\nText features and target labels verified successfully.")

Training text features: (28919,)
Testing text features: (6391,)
Training target labels: (28919,)
Testing target labels: (6391,)

Text features and target labels verified successfully.


#### Encoding Target Labels

In [4]:
# Initializing the target label encoder
label_encoder = LabelEncoder()

# Fitting the encoder on training labels
y_train_encoded = label_encoder.fit_transform(y_train)

# Transforming testing labels using the fitted encoder
y_test_encoded = label_encoder.transform(y_test)

# Creating the category-to-label mapping
label_mapping = pd.DataFrame({
    "Category": label_encoder.classes_,
    "Encoded_Label": range(len(label_encoder.classes_))
})

# Displaying the label mapping
display(label_mapping)

# Verifying the encoded target dimensions
print(f"Encoded training labels: {y_train_encoded.shape}")
print(f"Encoded testing labels: {y_test_encoded.shape}")
print(f"Number of encoded classes: {len(label_encoder.classes_)}")

,Category,Encoded_Label
0,CPUs,0
1,Digital Cameras,1
2,Dishwashers,2
3,Freezers,3
4,Fridge Freezers,4
5,Fridges,5
6,Microwaves,6
7,Mobile Phones,7
8,TVs,8
9,Washing Machines,9


Encoded training labels: (28919,)
Encoded testing labels: (6391,)
Number of encoded classes: 10


**Observations on Target Label Encoding:**

- All **10 product categories** were successfully encoded into numerical labels ranging from `0` to `9`.
- The encoded training and testing targets contain **28,919** and **6,391** records, respectively.
- A consistent category-to-label mapping was established using the training data and will be preserved for model evaluation and prediction.

#### Extracting Word-Level TF-IDF Features

In [5]:
# Initializing the word-level TF-IDF vectorizer
word_vectorizer = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    max_features=30000,
    sublinear_tf=True,
    dtype=np.float32
)

# Fitting the vectorizer on training product titles
X_train_word = word_vectorizer.fit_transform(X_train_text)

# Transforming testing titles using the fitted vectorizer
X_test_word = word_vectorizer.transform(X_test_text)

# Displaying the resulting feature dimensions
print(f"Training TF-IDF dimensions: {X_train_word.shape}")
print(f"Testing TF-IDF dimensions: {X_test_word.shape}")
print(f"Vocabulary size: {len(word_vectorizer.vocabulary_):,}")
print(f"Training matrix sparsity: {1 - X_train_word.nnz / (X_train_word.shape[0] * X_train_word.shape[1]):.2%}")
print(f"Testing matrix sparsity: {1 - X_test_word.nnz / (X_test_word.shape[0] * X_test_word.shape[1]):.2%}")

Training TF-IDF dimensions: (28919, 30000)
Testing TF-IDF dimensions: (6391, 30000)
Vocabulary size: 30,000
Training matrix sparsity: 99.95%
Testing matrix sparsity: 99.96%


**Observations on Word-Level TF-IDF Features:**

- Word-level TF-IDF generated **30,000 features** across 28,919 training records and 6,391 testing records.
- The vocabulary reached the configured `max_features` limit, indicating that the available word unigram and bigram vocabulary exceeds this threshold.
- Both matrices are highly sparse (**99.95% training** and **99.96% testing**), supporting the use of sparse representations for memory-efficient modeling.

#### Extracting Character-Level TF-IDF Features

In [6]:
# Initializing the character-level TF-IDF vectorizer
char_vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    min_df=2,
    max_features=50000,
    sublinear_tf=True,
    dtype=np.float32
)

# Fitting the vectorizer on training product titles
X_train_char = char_vectorizer.fit_transform(X_train_text)

# Transforming testing titles using the fitted vectorizer
X_test_char = char_vectorizer.transform(X_test_text)

# Displaying the resulting feature dimensions
print(f"Training character TF-IDF dimensions: {X_train_char.shape}")
print(f"Testing character TF-IDF dimensions: {X_test_char.shape}")
print(f"Character vocabulary size: {len(char_vectorizer.vocabulary_):,}")
print(f"Training matrix sparsity: {1 - X_train_char.nnz / (X_train_char.shape[0] * X_train_char.shape[1]):.2%}")
print(f"Testing matrix sparsity: {1 - X_test_char.nnz / (X_test_char.shape[0] * X_test_char.shape[1]):.2%}")

Training character TF-IDF dimensions: (28919, 50000)
Testing character TF-IDF dimensions: (6391, 50000)
Character vocabulary size: 50,000
Training matrix sparsity: 99.79%
Testing matrix sparsity: 99.80%


Why 50,000 features? Character n-grams generally produce a much larger vocabulary than word n-grams. This cap provides a practical starting point for our CPU-based models while keeping the matrices sparse.

**Observations on Character-Level TF-IDF Features:**

- Character-level TF-IDF generated **50,000 features** across 28,919 training records and 6,391 testing records.
- The vocabulary reached the configured feature limit, capturing character sequences from product names, model identifiers, and technical specifications.
- Both matrices remain highly sparse (**99.79% training** and **99.80% testing**), supporting memory-efficient storage and processing.

#### Combining Word-Level and Character-Level TF-IDF Features

In [8]:
# Combining word-level and character-level TF-IDF features
X_train_combined = hstack(
    [X_train_word, X_train_char],
    format="csr",
    dtype=np.float32
)

X_test_combined = hstack(
    [X_test_word, X_test_char],
    format="csr",
    dtype=np.float32
)

# Displaying combined feature dimensions
print(f"Combined training dimensions: {X_train_combined.shape}")
print(f"Combined testing dimensions: {X_test_combined.shape}")
print(f"Total combined features: {X_train_combined.shape[1]:,}")

# Calculating matrix sparsity
train_sparsity = 1 - (
    X_train_combined.nnz
    / (X_train_combined.shape[0] * X_train_combined.shape[1])
)

test_sparsity = 1 - (
    X_test_combined.nnz
    / (X_test_combined.shape[0] * X_test_combined.shape[1])
)

print(f"Combined training sparsity: {train_sparsity:.2%}")
print(f"Combined testing sparsity: {test_sparsity:.2%}")

# Verifying feature consistency
assert X_train_combined.shape[1] == X_test_combined.shape[1]
assert X_train_combined.shape[0] == len(y_train_encoded)
assert X_test_combined.shape[0] == len(y_test_encoded)

print("Combined feature matrices verified successfully.")

Combined training dimensions: (28919, 80000)
Combined testing dimensions: (6391, 80000)
Total combined features: 80,000
Combined training sparsity: 99.85%
Combined testing sparsity: 99.86%
Combined feature matrices verified successfully.


**Observations on Combined Word-Level and Character-Level TF-IDF Features:**

- The combined representation contains **80,000 features**, comprising 30,000 word-level and 50,000 character-level TF-IDF features.
- The training and testing matrices contain **28,919** and **6,391** records, respectively, with consistent feature dimensions.
- Both matrices remain highly sparse (**99.85% training** and **99.86% testing**), enabling memory-efficient storage and supporting sparse-compatible machine learning models.

#### Validating TF-IDF Feature Matrices

In [9]:
# Organizing feature matrices for validation
feature_matrices = {
    "Word TF-IDF": (X_train_word, X_test_word),
    "Character TF-IDF": (X_train_char, X_test_char),
    "Combined TF-IDF": (X_train_combined, X_test_combined)
}

# Validating each feature representation
for name, (train_matrix, test_matrix) in feature_matrices.items():
    assert train_matrix.shape[0] == len(y_train_encoded)
    assert test_matrix.shape[0] == len(y_test_encoded)
    assert train_matrix.shape[1] == test_matrix.shape[1]
    assert np.isfinite(train_matrix.data).all()
    assert np.isfinite(test_matrix.data).all()
    assert (train_matrix.data >= 0).all()
    assert (test_matrix.data >= 0).all()

    print(f"{name}: Validation passed")
    print(f"  Training shape: {train_matrix.shape}")
    print(f"  Testing shape: {test_matrix.shape}")
    print(f"  Training dtype: {train_matrix.dtype}")
    print(f"  Testing dtype: {test_matrix.dtype}")
    print()

print("All TF-IDF feature matrices validated successfully.")

Word TF-IDF: Validation passed
  Training shape: (28919, 30000)
  Testing shape: (6391, 30000)
  Training dtype: float32
  Testing dtype: float32

Character TF-IDF: Validation passed
  Training shape: (28919, 50000)
  Testing shape: (6391, 50000)
  Training dtype: float32
  Testing dtype: float32

Combined TF-IDF: Validation passed
  Training shape: (28919, 80000)
  Testing shape: (6391, 80000)
  Training dtype: float32
  Testing dtype: float32

All TF-IDF feature matrices validated successfully.


**Observations on TF-IDF Feature Validation:**

- All three feature representations passed dimensional consistency and numerical validity checks.
- No non-finite or negative TF-IDF values were identified in the training or testing matrices.
- All matrices use the memory-efficient `float32` data type and are ready for model development.